In [1]:
"""
第 1 步：导入必要的库和模块
"""
import os
import sys
import math
from types import SimpleNamespace
from collections import defaultdict
from datetime import datetime
from random import normalvariate
from IPython.display import display, clear_output, HTML

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm

# 导入项目模块
from drone_env import DroneSimulator
from drone_renderer_dynamic import DynamicSceneRenderer, DynamicDroneSimulator
from model import Model
from loss import DroneLoss

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
   显存: 9.6 GB


In [3]:
# ==============================================================================
# 第 2 步：训练配置
# ==============================================================================
args = SimpleNamespace(
    # ===== 训练参数 =====
    resume=None,                    # 恢复训练的模型路径 (例如 './checkpoints/xxx.pth')
    batch_size=16,                  # 批量大小
    num_iters=500,                  # 训练迭代次数 (Notebook 中可以设小一点先测试)
    timesteps=100,                  # 每次迭代的模拟步数
    lr=1e-3,                        # 学习率
    grad_decay=0.4,                 # 梯度衰减系数
    ctl_dt=1/15,                    # 控制时间步长 (秒)
    
    # ===== 损失函数权重 =====
    coef_v=1.0,                     # 速度跟踪损失权重
    coef_speed=0.0,                 # 速度损失权重
    coef_v_pred=2.0,                # 速度预测损失权重
    coef_collide=2.0,               # 碰撞损失权重
    coef_obj_avoidance=1.5,         # 障碍物回避损失权重
    coef_d_acc=0.01,                # 加速度正则化权重
    coef_d_jerk=0.001,              # 加加速度正则化权重
    coef_d_snap=0.0,                # snap正则化权重
    coef_ground_affinity=0.0,       # 地面亲和损失权重
    coef_bias=0.0,                  # 方向偏差损失权重
    window_size=30,                 # 速度平均窗口大小
    
    # ===== 渲染参数 =====
    cam_angle=10,                   # 相机俯仰角 (度)
    image_height=48,                # 图像高度
    image_width=64,                 # 图像宽度
    mesh_path='./data/sample/sample4.obj',  # 静态场景网格路径
    num_samples=100000,             # 点云采样数
    
    # ===== 动态障碍物参数 =====
    num_dynamic_obs=3,              # 动态障碍物数量
    obs_pos_range=3.0,              # 障碍物位置范围 [-range, range]
    obs_vel_range=0.3,              # 障碍物速度范围 [-range, range]
    obs_scale_min=0.2,              # 障碍物最小缩放
    obs_scale_max=0.6,              # 障碍物最大缩放
    randomize_each_episode=True,    # 每个 episode 随机化障碍物
    
    # ===== 无人机物理参数 =====
    margin_min=0.7,                 # 安全半径最小值
    margin_max=1.0,                 # 安全半径最大值
    init_p_range=2.0,               # 初始位置范围
    noise_std=0.04,                 # 环境扰动噪声
    yaw_inertia=5.0,                # 偏航惯性
    yaw_ctl_delay=12.0,             # 偏航控制延迟
    pitch_ctl_delay=12.0,           # 俯仰控制延迟
    airmode_coef=0.5,               # Airmode 系数
    enable_airmode=True,            # 启用 Airmode
    disable_airmode=False,          # 禁用 Airmode
    
    # ===== 模型参数 =====
    no_odom=True,                  # 不使用里程计速度
    yaw_drift=False,                # 启用航向漂移
    debug=False,                    # 启用 anomaly detection
    
    # ===== 保存参数 =====
    save_dir='./checkpoints',
    log_dir='./logs'
)

print("📋 训练配置:")
print("=" * 50)
for k, v in vars(args).items():
    print(f"  {k}: {v}")

📋 训练配置:
  resume: None
  batch_size: 16
  num_iters: 500
  timesteps: 100
  lr: 0.001
  grad_decay: 0.4
  ctl_dt: 0.06666666666666667
  coef_v: 1.0
  coef_speed: 0.0
  coef_v_pred: 2.0
  coef_collide: 2.0
  coef_obj_avoidance: 1.5
  coef_d_acc: 0.01
  coef_d_jerk: 0.001
  coef_d_snap: 0.0
  coef_ground_affinity: 0.0
  coef_bias: 0.0
  window_size: 30
  cam_angle: 10
  image_height: 48
  image_width: 64
  mesh_path: ./data/sample/sample4.obj
  num_samples: 100000
  num_dynamic_obs: 3
  obs_pos_range: 3.0
  obs_vel_range: 0.3
  obs_scale_min: 0.2
  obs_scale_max: 0.6
  randomize_each_episode: True
  margin_min: 0.7
  margin_max: 1.0
  init_p_range: 2.0
  noise_std: 0.04
  yaw_inertia: 5.0
  yaw_ctl_delay: 12.0
  pitch_ctl_delay: 12.0
  airmode_coef: 0.5
  enable_airmode: True
  disable_airmode: False
  no_odom: True
  yaw_drift: False
  debug: False
  save_dir: ./checkpoints
  log_dir: ./logs


In [4]:
# ==============================================================================
# 第 3 步：初始化训练环境
# ==============================================================================

class DynamicTrainerNotebook:
    """Notebook 专用的动态场景训练器"""
    
    def __init__(self, args):
        self.args = args
        self.device = device
        self.ctl_dt = args.ctl_dt
        
        # 处理 airmode
        enable_airmode = args.enable_airmode and not args.disable_airmode
        
        # 初始化基础环境
        print("🔧 初始化基础仿真环境...")
        self.base_env = DroneSimulator(
            batch_size=args.batch_size,
            dt=self.ctl_dt,
            mesh_path=args.mesh_path,
            image_size=(args.image_height, args.image_width),
            device=self.device,
            enable_airmode=enable_airmode,
            noise_std=args.noise_std,
            grad_decay=args.grad_decay,
            yaw_inertia=args.yaw_inertia,
            yaw_ctl_delay=args.yaw_ctl_delay,
            pitch_ctl_delay=args.pitch_ctl_delay,
            airmode_coef=args.airmode_coef,
            init_p_range=args.init_p_range,
            init_margin_range=(args.margin_min, args.margin_max),
            num_samples=args.num_samples
        )
        
        # 创建动态场景渲染器
        print("🎬 创建动态场景渲染器...")
        self.dynamic_renderer = DynamicSceneRenderer(
            static_mesh_path=args.mesh_path,
            device=self.device,
            image_size=(args.image_height, args.image_width),
            focal_length=500.0,
            num_samples=args.num_samples
        )
        
        # 替换渲染器
        self.base_env.renderer = self.dynamic_renderer
        self.env = DynamicDroneSimulator(self.base_env, self.dynamic_renderer)
        
        # 初始化模型
        print("🧠 初始化神经网络模型...")
        dim_obs = 7 if args.no_odom else 10
        self.model = Model(dim_obs=dim_obs, dim_action=6).to(self.device)
        
        if args.resume:
            state_dict = torch.load(args.resume, map_location=self.device)
            self.model.load_state_dict(state_dict, strict=False)
            print(f"   ✅ 加载预训练模型: {args.resume}")
        
        # 优化器
        from torch.optim import AdamW
        from torch.optim.lr_scheduler import CosineAnnealingLR
        
        self.optimizer = AdamW(self.model.parameters(), lr=args.lr)
        self.scheduler = CosineAnnealingLR(self.optimizer, args.num_iters, eta_min=args.lr * 0.01)
        
        # 损失函数
        self.losser = DroneLoss(
            coef_v=args.coef_v,
            coef_speed=args.coef_speed,
            coef_v_pred=args.coef_v_pred,
            coef_collide=args.coef_collide,
            coef_obj_avoidance=args.coef_obj_avoidance,
            coef_d_acc=args.coef_d_acc,
            coef_d_jerk=args.coef_d_jerk,
            coef_d_snap=args.coef_d_snap,
            coef_ground_affinity=args.coef_ground_affinity,
            coef_bias=args.coef_bias,
            ctl_dt=self.ctl_dt,
            window_size=args.window_size
        )
        
        # 重力向量
        self.g_std = torch.tensor([0.0, 0.0, -9.80665], device=self.device)
        
        # 可视化数据
        self.last_vis_data = None
        
        print(f"\n✅ 初始化完成!")
        print(f"   模型参数量: {sum(p.numel() for p in self.model.parameters()):,}")
    
    def _compute_local_R(self):
        """计算局部坐标系旋转矩阵"""
        fwd = self.base_env.R[:, :, 0].clone()
        up = torch.zeros_like(fwd)
        fwd[:, 2] = 0
        up[:, 2] = 1
        fwd_norm = torch.norm(fwd, p=2, dim=-1, keepdim=True)
        fwd = torch.where(fwd_norm > 1e-6, fwd / (fwd_norm + 1e-8), 
                          torch.tensor([1.0, 0.0, 0.0], device=self.device).expand_as(fwd))
        R = torch.stack([fwd, torch.linalg.cross(up, fwd), up], -1)
        return R
    
    def _randomize_obstacles(self):
        """随机化动态障碍物"""
        args = self.args
        self.dynamic_renderer.randomize_obstacles(
            num_obstacles=args.num_dynamic_obs,
            position_range=(-args.obs_pos_range, args.obs_pos_range),
            velocity_range=(-args.obs_vel_range, args.obs_vel_range),
            scale_range=(args.obs_scale_min, args.obs_scale_max)
        )
    
    def run_episode(self, record_vis=False):
        """运行一个 episode"""
        args = self.args
        B = args.batch_size
        
        # 重置
        self.base_env.reset()
        self.model.reset()
        
        if args.randomize_each_episode:
            self._randomize_obstacles()
        
        # 历史记录
        p_history, v_history, target_v_history = [], [], []
        vec_to_pt_history, v_preds = [], []
        vid, rgb_vid, obs_pos_history = [], [], []
        
        h = None
        act_lag = 1
        initial_act = self.base_env.act_curr.clone()
        act_buffer = [initial_act.clone() for _ in range(act_lag + 1)]
        
        p_target = torch.rand(B, 3, device=self.device) * 18.0 - 9.0
        target_v_raw = p_target - self.base_env.p
        max_speed = 0.75 + 2.5 * torch.rand((B, 1), device=self.device)
        thr_est_error = 1.0 + 0.1 * torch.randn((B, 1), device=self.device)
        
        for t in range(args.timesteps):
            current_dt = normalvariate(self.ctl_dt, self.ctl_dt * 0.1)
            self.dynamic_renderer.step_obstacles(current_dt)
            
            with torch.no_grad():
                rgb, depth = self.base_env.render(
                    camera_pitch=args.cam_angle,
                    return_tensor=True,
                    return_rgb=True,
                    return_depth=True,
                    dt=current_dt
                )
            depth = depth.requires_grad_(False)
            
            p_history.append(self.base_env.p.clone())
            v_history.append(self.base_env.v.clone())
            vec_to_pt_history.append(self.base_env.vec_to_obj())
            
            if record_vis:
                vid.append(depth[0].cpu())
                rgb_vid.append(rgb[0].cpu())
                obs_pos = [obs.position.cpu().numpy() for obs in self.dynamic_renderer.dynamic_obstacles]
                obs_pos_history.append(np.array(obs_pos) if obs_pos else np.array([]))
            
            target_v_raw = p_target - self.base_env.p.detach()
            self.base_env.step(act_cmd=act_buffer[t], target_pos_vector=target_v_raw, dt=current_dt)
            
            R_local = self._compute_local_R()
            target_v_norm = torch.norm(target_v_raw, p=2, dim=-1, keepdim=True)
            target_v_unit = target_v_raw / (target_v_norm + 1e-6)
            target_v = target_v_unit * torch.minimum(target_v_norm, max_speed)
            target_v_history.append(target_v)
            
            target_v_local = torch.squeeze(target_v[:, None] @ R_local, 1)
            local_v = torch.squeeze(self.base_env.v[:, None] @ R_local, 1)
            
            state_parts = [target_v_local, self.base_env.R[:, 2], self.base_env.margin[:, None]]
            if not args.no_odom:
                state_parts.insert(0, local_v)
            state = torch.cat(state_parts, dim=-1)
            
            x = depth.clamp(0.3, 24.0)
            x = 3.0 / x - 0.6
            x = x + torch.randn_like(x) * 0.02
            x = F.max_pool2d(x[:, None], kernel_size=4, stride=4)
            
            act_raw, _, h = self.model(x, state, h)
            act_reshaped = act_raw.reshape(B, 2, 3).permute(0, 2, 1)
            act_world = R_local @ act_reshaped
            a_pred, v_pred = act_world.unbind(-1)
            
            v_preds.append(v_pred)
            act = (a_pred - v_pred - self.g_std) * thr_est_error + self.g_std
            act_buffer.append(act)
        
        # 堆叠
        p_history = torch.stack(p_history)
        v_history = torch.stack(v_history)
        target_v_history = torch.stack(target_v_history)
        vec_to_pt_history = torch.stack(vec_to_pt_history)
        v_preds = torch.stack(v_preds)
        act_buffer_stacked = torch.stack(act_buffer)
        
        # 损失
        loss, metrics = self.losser.forward(
            p_history=p_history, v_history=v_history,
            target_vel_history=target_v_history, act_history=act_buffer_stacked,
            vec_to_obj_history=vec_to_pt_history, v_preds=v_preds,
            env_margin=self.base_env.margin, env_g_std=self.g_std
        )
        
        with torch.no_grad():
            distance = torch.norm(vec_to_pt_history, 2, -1) - self.base_env.margin
            speed_hist = v_history.norm(2, -1)
            avg_speed = speed_hist.mean(0)
            success = torch.all(distance.flatten(0, 1) > 0, 0)
            metrics['success_rate'] = success.float().mean()
            metrics['avg_speed'] = avg_speed.mean()
            metrics['ar'] = (success.float() * avg_speed).mean()
        
        vis_data = {
            'p_history': p_history, 'v_history': v_history,
            'act_history': act_buffer_stacked, 'vid': vid, 'rgb_vid': rgb_vid,
            'obstacle_positions': obs_pos_history, 'target_pos': p_target
        }
        
        if record_vis:
            self.last_vis_data = vis_data
        
        return loss, metrics, vis_data

# 创建训练器
trainer = DynamicTrainerNotebook(args)

🔧 初始化基础仿真环境...
Loading mesh from: ./data/sample/sample4.obj
🎬 创建动态场景渲染器...
🧠 初始化神经网络模型...

✅ 初始化完成!
   模型参数量: 513,920


In [ ]:
# ==============================================================================
# 第 4 步：训练主循环 (带实时可视化)
# ==============================================================================

def to_float(x):
    """将 tensor 转换为 Python float"""
    if torch.is_tensor(x):
        return x.detach().cpu().item()
    return float(x)

# 训练记录
history = defaultdict(list)

# 设置绘图
plt.ion()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('🚁 Drone Dynamic Training Progress', fontsize=14)

def update_plots(history, iteration):
    """更新训练曲线图"""
    for ax in axes.flat:
        ax.clear()
    
    iters = range(len(history['loss']))
    
    # 总损失
    axes[0, 0].plot(iters, history['loss'], 'b-', alpha=0.3)
    if len(history['loss']) >= 10:
        smooth = np.convolve(history['loss'], np.ones(10)/10, mode='valid')
        axes[0, 0].plot(range(9, len(history['loss'])), smooth, 'b-', linewidth=2)
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 速度损失
    axes[0, 1].plot(iters, history['loss_v'], 'g-', alpha=0.5, label='loss_v')
    axes[0, 1].plot(iters, history['loss_v_pred'], 'r-', alpha=0.5, label='loss_v_pred')
    axes[0, 1].set_title('Velocity Losses')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 碰撞损失
    axes[0, 2].plot(iters, history['loss_collide'], 'r-', alpha=0.5, label='collide')
    axes[0, 2].plot(iters, history['loss_obj_avoidance'], 'orange', alpha=0.5, label='avoidance')
    axes[0, 2].set_title('Collision Losses')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # 成功率
    axes[1, 0].plot(iters, history['success_rate'], 'g-')
    axes[1, 0].set_title('Success Rate')
    axes[1, 0].set_ylim([0, 1.05])
    axes[1, 0].axhline(y=1.0, color='r', linestyle='--', alpha=0.3)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 平均速度
    axes[1, 1].plot(iters, history['avg_speed'], 'b-')
    axes[1, 1].set_title('Average Speed (m/s)')
    axes[1, 1].grid(True, alpha=0.3)
    
    # AR
    axes[1, 2].plot(iters, history['ar'], 'purple')
    axes[1, 2].set_title('AR (Success × Speed)')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    fig.canvas.draw()
    fig.canvas.flush_events()

# 训练主循环
print("🚀 开始训练...")
pbar = tqdm(range(args.num_iters), desc="Training")

for i in pbar:
    # 是否记录可视化 (每100次记录一次)
    record_vis = (i + 1) % 100 == 0
    
    # 运行 episode
    loss, metrics, _ = trainer.run_episode(record_vis=record_vis)
    
    if torch.isnan(loss):
        print(f"\n⚠️ 损失出现 NaN 在迭代 {i}, 停止训练...")
        break
    
    # 反向传播
    trainer.optimizer.zero_grad()
    loss.backward()
    trainer.optimizer.step()
    trainer.scheduler.step()
    
    # 记录历史
    history['loss'].append(loss.item())
    history['loss_v'].append(to_float(metrics.get('loss_v', 0)))
    history['loss_v_pred'].append(to_float(metrics.get('loss_v_pred', 0)))
    history['loss_collide'].append(to_float(metrics.get('loss_collide', 0)))
    history['loss_obj_avoidance'].append(to_float(metrics.get('loss_obj_avoidance', 0)))
    history['success_rate'].append(to_float(metrics.get('success_rate', 0)))
    history['avg_speed'].append(to_float(metrics.get('avg_speed', 0)))
    history['ar'].append(to_float(metrics.get('ar', 0)))
    
    # 更新进度条
    pbar.set_postfix({
        'loss': f"{loss.item():.3f}",
        'sr': f"{to_float(metrics.get('success_rate', 0)):.2f}",
        'ar': f"{to_float(metrics.get('ar', 0)):.2f}"
    })
    
    # 定期更新图表
    if (i + 1) % 20 == 0:
        update_plots(history, i)

update_plots(history, args.num_iters)
plt.ioff()
print("\n✅ 训练完成!")

🚀 开始训练...


Training:   0%|          | 0/500 [00:00<?, ?it/s]

/tmp/ipykernel_70740/2890902526.py:67: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_70740/2890902526.py:68: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()


In [ ]:
# ==============================================================================
# 第 5 步：可视化无人机在场景中做了什么
# ==============================================================================

# 运行一个 episode 并记录所有数据
print("🔍 运行可视化 Episode...")
with torch.no_grad():
    trainer.model.eval()
    _, metrics, vis_data = trainer.run_episode(record_vis=True)
    trainer.model.train()

# 提取数据
p_np = vis_data['p_history'][:, 0].cpu().numpy()  # 第一个样本
v_np = vis_data['v_history'][:, 0].cpu().numpy()
target_pos = vis_data['target_pos'][0].cpu().numpy()
depth_frames = vis_data['vid']
rgb_frames = vis_data['rgb_vid']
obs_positions = vis_data['obstacle_positions']

T = p_np.shape[0]
print(f"\n📊 Episode 统计:")
print(f"  时间步数: {T}")
print(f"  成功率: {metrics['success_rate']:.2%}")
print(f"  平均速度: {metrics['avg_speed']:.2f} m/s")
print(f"  AR: {metrics['ar']:.3f}")

In [ ]:
# ==============================================================================
# 第 6 步：3D 轨迹可视化
# ==============================================================================

fig = plt.figure(figsize=(16, 6))

# 子图1：3D 轨迹 + 障碍物
ax1 = fig.add_subplot(131, projection='3d')

# 无人机轨迹
scatter = ax1.scatter(p_np[:, 0], p_np[:, 1], p_np[:, 2], 
                      c=np.arange(T), cmap='viridis', s=10, alpha=0.8)
ax1.plot(p_np[:, 0], p_np[:, 1], p_np[:, 2], 'b-', alpha=0.3, linewidth=1)

# 起点和终点
ax1.scatter(*p_np[0], color='green', s=150, marker='o', label='Start', edgecolors='black')
ax1.scatter(*p_np[-1], color='red', s=150, marker='x', label='End', linewidths=3)
ax1.scatter(*target_pos, color='gold', s=200, marker='*', label='Target', edgecolors='black')

# 障碍物位置（最后时刻）
if len(obs_positions) > 0 and len(obs_positions[-1]) > 0:
    for obs in obs_positions[-1]:
        ax1.scatter(*obs, color='orange', s=300, marker='s', alpha=0.7, label='Obstacle')

ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title('3D Drone Trajectory')
ax1.legend()
plt.colorbar(scatter, ax=ax1, label='Time step', shrink=0.5)

# 子图2：XY 平面俯视图
ax2 = fig.add_subplot(132)
ax2.plot(p_np[:, 0], p_np[:, 1], 'b-', linewidth=2, alpha=0.7)
ax2.scatter(p_np[0, 0], p_np[0, 1], color='green', s=100, marker='o', label='Start')
ax2.scatter(p_np[-1, 0], p_np[-1, 1], color='red', s=100, marker='x', label='End')
ax2.scatter(target_pos[0], target_pos[1], color='gold', s=150, marker='*', label='Target')

if len(obs_positions) > 0 and len(obs_positions[-1]) > 0:
    for obs in obs_positions[-1]:
        circle = plt.Circle((obs[0], obs[1]), 0.3, color='orange', alpha=0.5)
        ax2.add_patch(circle)

ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('Top View (XY Plane)')
ax2.axis('equal')
ax2.grid(True, alpha=0.3)
ax2.legend()

# 子图3：XZ 平面侧视图
ax3 = fig.add_subplot(133)
ax3.plot(p_np[:, 0], p_np[:, 2], 'b-', linewidth=2, alpha=0.7)
ax3.scatter(p_np[0, 0], p_np[0, 2], color='green', s=100, marker='o', label='Start')
ax3.scatter(p_np[-1, 0], p_np[-1, 2], color='red', s=100, marker='x', label='End')
ax3.scatter(target_pos[0], target_pos[2], color='gold', s=150, marker='*', label='Target')
ax3.axhline(y=0, color='brown', linestyle='--', alpha=0.5, label='Ground')

ax3.set_xlabel('X (m)')
ax3.set_ylabel('Z (m)')
ax3.set_title('Side View (XZ Plane)')
ax3.grid(True, alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 第 7 步：无人机看到了什么 - 深度图和 RGB 图像可视化
# ==============================================================================

if len(depth_frames) > 0:
    n_frames = min(8, len(depth_frames))
    frame_indices = np.linspace(0, len(depth_frames)-1, n_frames, dtype=int)
    
    fig, axes = plt.subplots(2, n_frames, figsize=(n_frames * 2.5, 6))
    fig.suptitle('🎥 无人机视觉观测 (深度 + RGB)', fontsize=14)
    
    for idx, frame_idx in enumerate(frame_indices):
        # 深度图
        depth_img = depth_frames[frame_idx].numpy()
        axes[0, idx].imshow(depth_img, cmap='viridis')
        axes[0, idx].set_title(f't={frame_idx}')
        axes[0, idx].axis('off')
        
        # RGB 图
        if len(rgb_frames) > frame_idx:
            rgb_img = rgb_frames[frame_idx].numpy()
            rgb_img = (rgb_img - rgb_img.min()) / (rgb_img.max() - rgb_img.min() + 1e-8)
            axes[1, idx].imshow(rgb_img)
        axes[1, idx].axis('off')
    
    axes[0, 0].set_ylabel('Depth', fontsize=12)
    axes[1, 0].set_ylabel('RGB', fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 没有记录到视觉帧数据")

In [ ]:
# ==============================================================================
# 第 8 步：状态历史可视化 (位置、速度、动作)
# ==============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
time_axis = np.arange(T) * args.ctl_dt  # 转换为真实时间

# 位置
axes[0].plot(time_axis, p_np[:, 0], 'r-', label='x', linewidth=1.5)
axes[0].plot(time_axis, p_np[:, 1], 'g-', label='y', linewidth=1.5)
axes[0].plot(time_axis, p_np[:, 2], 'b-', label='z', linewidth=1.5)
axes[0].axhline(y=target_pos[0], color='r', linestyle='--', alpha=0.3)
axes[0].axhline(y=target_pos[1], color='g', linestyle='--', alpha=0.3)
axes[0].axhline(y=target_pos[2], color='b', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Position (m)')
axes[0].set_title('Position History')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 速度
axes[1].plot(time_axis, v_np[:, 0], 'r-', label='vx', linewidth=1.5)
axes[1].plot(time_axis, v_np[:, 1], 'g-', label='vy', linewidth=1.5)
axes[1].plot(time_axis, v_np[:, 2], 'b-', label='vz', linewidth=1.5)
speed = np.linalg.norm(v_np, axis=1)
axes[1].plot(time_axis, speed, 'k--', label='|v|', linewidth=2)
axes[1].set_ylabel('Velocity (m/s)')
axes[1].set_title('Velocity History')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 动作
act_np = vis_data['act_history'][:T, 0].cpu().numpy()
axes[2].plot(time_axis, act_np[:, 0], 'r-', label='ax', linewidth=1.5)
axes[2].plot(time_axis, act_np[:, 1], 'g-', label='ay', linewidth=1.5)
axes[2].plot(time_axis, act_np[:, 2], 'b-', label='az', linewidth=1.5)
axes[2].axhline(y=-9.8, color='gray', linestyle='--', alpha=0.5, label='gravity')
axes[2].set_ylabel('Acceleration (m/s²)')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Action (Acceleration) History')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 第 9 步：障碍物运动轨迹可视化
# ==============================================================================

if len(obs_positions) > 0 and len(obs_positions[0]) > 0:
    fig = plt.figure(figsize=(12, 5))
    
    ax1 = fig.add_subplot(121, projection='3d')
    
    # 绘制每个障碍物的轨迹
    n_obs = len(obs_positions[0])
    colors = plt.cm.Set1(np.linspace(0, 1, n_obs))
    
    for obs_idx in range(n_obs):
        obs_traj = np.array([obs_positions[t][obs_idx] for t in range(len(obs_positions)) if len(obs_positions[t]) > obs_idx])
        if len(obs_traj) > 0:
            ax1.plot(obs_traj[:, 0], obs_traj[:, 1], obs_traj[:, 2], 
                    color=colors[obs_idx], linewidth=2, alpha=0.7, label=f'Obs {obs_idx+1}')
            ax1.scatter(*obs_traj[0], color=colors[obs_idx], marker='o', s=100)
            ax1.scatter(*obs_traj[-1], color=colors[obs_idx], marker='x', s=100)
    
    # 绘制无人机轨迹
    ax1.plot(p_np[:, 0], p_np[:, 1], p_np[:, 2], 'b-', linewidth=2, alpha=0.5, label='Drone')
    
    ax1.set_xlabel('X (m)')
    ax1.set_ylabel('Y (m)')
    ax1.set_zlabel('Z (m)')
    ax1.set_title('Obstacle & Drone Trajectories')
    ax1.legend()
    
    # 时间序列图
    ax2 = fig.add_subplot(122)
    
    # 计算无人机到各障碍物的距离
    for obs_idx in range(n_obs):
        distances = []
        for t in range(min(T, len(obs_positions))):
            if len(obs_positions[t]) > obs_idx:
                dist = np.linalg.norm(p_np[t] - obs_positions[t][obs_idx])
                distances.append(dist)
        if distances:
            ax2.plot(distances, color=colors[obs_idx], linewidth=2, label=f'To Obs {obs_idx+1}')
    
    ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Safety margin')
    ax2.set_xlabel('Time step')
    ax2.set_ylabel('Distance (m)')
    ax2.set_title('Distance to Obstacles')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 没有动态障碍物数据")

In [ ]:
# ==============================================================================
# 第 10 步：保存模型
# ==============================================================================

import json

os.makedirs(args.save_dir, exist_ok=True)
os.makedirs(args.log_dir, exist_ok=True)

# 保存模型
model_path = os.path.join(args.save_dir, 'model_dynamic_notebook.pth')
torch.save(trainer.model.state_dict(), model_path)
print(f"✅ 模型已保存到: {model_path}")

# 保存训练历史
history_path = os.path.join(args.log_dir, 'training_history_dynamic.json')
with open(history_path, 'w') as f:
    json.dump(dict(history), f, indent=2)
print(f"📊 训练历史已保存到: {history_path}")

# 保存配置
config_path = os.path.join(args.log_dir, 'training_config_dynamic.json')
with open(config_path, 'w') as f:
    json.dump(vars(args), f, indent=2)
print(f"⚙️ 训练配置已保存到: {config_path}")

In [ ]:
# ==============================================================================
# 第 11 步：创建轨迹动画 (可选)
# ==============================================================================

from IPython.display import HTML

def create_trajectory_animation(p_np, obs_positions, target_pos, interval=50):
    """创建 3D 轨迹动画"""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 设置坐标轴范围
    all_points = p_np
    margin = 2
    ax.set_xlim([all_points[:, 0].min() - margin, all_points[:, 0].max() + margin])
    ax.set_ylim([all_points[:, 1].min() - margin, all_points[:, 1].max() + margin])
    ax.set_zlim([all_points[:, 2].min() - margin, all_points[:, 2].max() + margin])
    
    # 初始化图形元素
    line, = ax.plot([], [], [], 'b-', linewidth=2, alpha=0.5)
    point, = ax.plot([], [], [], 'ro', markersize=10)
    ax.scatter(*target_pos, color='gold', s=200, marker='*', label='Target')
    
    # 障碍物散点
    obs_scatter = ax.scatter([], [], [], color='orange', s=200, marker='s', alpha=0.7)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title('Drone Navigation Animation')
    
    def init():
        line.set_data([], [])
        line.set_3d_properties([])
        point.set_data([], [])
        point.set_3d_properties([])
        return line, point
    
    def animate(frame):
        # 更新轨迹
        line.set_data(p_np[:frame+1, 0], p_np[:frame+1, 1])
        line.set_3d_properties(p_np[:frame+1, 2])
        
        # 更新当前位置
        point.set_data([p_np[frame, 0]], [p_np[frame, 1]])
        point.set_3d_properties([p_np[frame, 2]])
        
        # 更新障碍物位置
        if frame < len(obs_positions) and len(obs_positions[frame]) > 0:
            obs = np.array(obs_positions[frame])
            obs_scatter._offsets3d = (obs[:, 0], obs[:, 1], obs[:, 2])
        
        ax.set_title(f'Drone Navigation (t={frame})')
        return line, point
    
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=len(p_np), interval=interval, blit=False)
    plt.close()
    return anim

# 创建动画
print("🎬 创建轨迹动画...")
anim = create_trajectory_animation(p_np, obs_positions, target_pos, interval=50)

# 在 notebook 中显示
HTML(anim.to_jshtml())

---

## 📌 使用说明

1. **运行顺序**：按单元格顺序执行
2. **调整参数**：在第 2 步修改 `args` 配置
3. **恢复训练**：设置 `args.resume = './checkpoints/xxx.pth'`
4. **增加迭代次数**：修改 `args.num_iters`
5. **调整动态障碍物**：修改 `num_dynamic_obs`, `obs_pos_range` 等参数

## 📊 输出说明

- **3D 轨迹图**：无人机飞行路径，带起点、终点、目标点和障碍物
- **深度图序列**：无人机摄像头看到的深度信息
- **RGB 图序列**：无人机摄像头看到的颜色图像
- **状态历史**：位置、速度、加速度随时间变化
- **障碍物轨迹**：动态障碍物的移动路径和无人机与它们的距离